Nigerian Wiki 

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import os
from pydantic import BaseModel,  Field
from dotenv import load_dotenv

import gradio as gr
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np
import plotly.graph_objects as go
import gradio as gr

In [ ]:
# Initialize OpenAI and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
MODEL = "all-MiniLM-L6-v2"


db_name = "vector_db"
collection_name = "sites"


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = WebBaseLoader(["https://en.wikipedia.org/wiki/Nigeria",
                        "https://en.wikipedia.org/wiki/Economic_history_of_Nigeria",
                        "https://en.wikipedia.org/wiki/Nigerian_naira",
                        "https://en.wikipedia.org/wiki/Nigerian_Exchange_Group",
                        "https://en.wikipedia.org/wiki/Petroleum_industry_in_Nigeria",
                        "https://en.wikipedia.org/wiki/Energy_in_Nigeria",
                        "https://en.wikipedia.org/wiki/Telecommunications_in_Nigeria",
                        "https://en.wikipedia.org/wiki/Tourism_in_Nigeria",
                        "https://en.wikipedia.org/wiki/Transport_in_Nigeria",
                        "https://en.wikipedia.org/wiki/Agriculture_in_Nigeria",
                        "https://en.wikipedia.org/wiki/Central_Bank_of_Nigeria",
                        "https://en.wikipedia.org/wiki/History_of_Nigeria",
                        "https://en.wikipedia.org/wiki/Demographics_of_Nigeria"])
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)
chunks 

In [ ]:

embeddings = HuggingFaceEmbeddings(model_name=MODEL)

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)




In [ ]:
chunks[0]


In [ ]:
collection = vectorstore._client.get_or_create_collection(collection_name)
def save_chunks(chunks):
    try:
        vectorstore.delete_collection(collection_name)
    except Exception:
        pass  # Collection doesn't exist, that's fine

    texts = [chunk.page_content for chunk in chunks]
    vectors = embeddings.embed_documents(texts)  # returns a list of vectors directly

    

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

save_chunks(chunks)



In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(RankOrder)
    response = llm.invoke(messages)
    order = response.order
    # order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:


from langchain_core.documents import Document


RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = embeddings.embed_query(question)
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = [
        Document(page_content=doc, metadata=meta)
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]
    return chunks

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant for everything Nigeria.
You are chatting with a user about Nigeria.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def answer_question(question: str, history: list[tuple[str, str]]):
    chunks = fetch_context(question)

    print(chunks)

    # Convert gradio history (tuples) into OpenAI format
    formatted_history = []
    for user_msg, bot_msg in history:
        formatted_history.append({"role": "user", "content": user_msg})
        formatted_history.append({"role": "assistant", "content": bot_msg})

    messages = make_rag_messages(question, formatted_history, chunks)

    llm = ChatOpenAI(model="gpt-4o-mini")
    response = llm.invoke(messages)

    return response.content

In [ ]:
def chat(message, history):
    if not message.strip():
        return history, ""
    answer = answer_question(message, history)
    history = history + [(message, answer)]
    return history, ""


with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="green"),
    title="🇳🇬 Nigeria RAG Agent",
    css="""
        #header { text-align: center; padding: 20px 0 10px; }
        #header h1 { font-size: 2.2em; font-weight: 800; color: #1a6e2e; }
        #header p { color: #555; font-size: 1.05em; }
        #status-box { border-radius: 8px; }
        .chatbot { min-height: 420px; }
        #load-btn { background: #1a6e2e !important; color: white !important; font-weight: 600; }
    """
) as demo:

    with gr.Column(elem_id="header"):
        gr.HTML("""
            <h1>🇳🇬 Nigeria Expert Agent</h1>
            <p>Ask anything about Nigeria — history, culture, economy, geography, politics and more.</p>
        """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Setup")
            load_btn = gr.Button("🔄 Load Knowledge Base", elem_id="load-btn", size="lg")
            status = gr.Textbox(
                label="Status",
                value="Click the button to load the Nigeria knowledge base.",
                interactive=False,
                elem_id="status-box",
                lines=2
            )

            gr.Markdown("### 💡 Try asking...")
           

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Chat with Nigeria Expert",
                elem_classes=["chatbot"],
                bubble_full_width=False,
                avatar_images=(None, "https://flagcdn.com/w40/ng.png"),
                height=480,
            )

            with gr.Column(scale=1):
                    msg = gr.Textbox(
                        placeholder="Ask anything about Nigeria...",
                        show_label=False,
                        scale=5,
                        container=False,
                    )
                    send_btn = gr.Button("Send 🚀", scale=1, variant="primary")
                    gr.Examples(
                examples=[
                    ["What is Nigeria's GDP and main industries?"],
                    ["Tell me about Nigerian culture and traditions"],
                    ["What are the major ethnic groups in Nigeria?"],
                    ["What is the history of Nigeria's independence?"],
                    ["What is Nigeria's current political system?"],
                    ["What are the most popular Nigerian foods?"],
                ],
                inputs=[msg],  # link to the textbox
                label=""
            )
            

            clear_btn = gr.Button("🗑️ Clear Chat", size="sm", variant="secondary")

    
    send_btn.click(fn=chat, inputs=[msg, chatbot], outputs=[chatbot, msg])
    msg.submit(fn=chat, inputs=[msg, chatbot], outputs=[chatbot, msg])
    clear_btn.click(fn=lambda: ([], ""), outputs=[chatbot, msg])

demo.launch()